In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

# Upload file
uploaded = files.upload()
import pandas as pd

# Get file name
file_name = list(uploaded.keys())[0]

# Create DataFrame
df = pd.read_excel(file_name)

# Now this will work
print(df.columns)

# Create input and target
df["input_text"] = df["description"]
df["target_text"] = df["title"]

# Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["input_text"])

X = tokenizer.texts_to_sequences(df["input_text"])
X = pad_sequences(X)

# Encode titles
label_tokenizer = Tokenizer()
label_tokenizer.fit_on_texts(df["target_text"])

y = label_tokenizer.texts_to_sequences(df["target_text"])
y = pad_sequences(y)

# Expand dimension for sparse categorical loss
y = np.expand_dims(y, -1)

# Build Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed

# define vocab sizes
vocab_size = len(tokenizer.word_index) + 1
title_vocab_size = len(label_tokenizer.word_index) + 1

model = Sequential([
    Embedding(vocab_size, 72),
    LSTM(128, return_sequences=True),
    TimeDistributed(Dense(title_vocab_size, activation="softmax"))
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
y[:,0]

def generate_title(text):

    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=X.shape[1])

    pred = model.predict(seq)[0]

    index_to_word = {index: word for word, index in label_tokenizer.word_index.items()}

    result = []
    last_word = None

    for timestep in pred:
        predicted_index = timestep.argmax()

        if predicted_index == 0:
            continue

        word = index_to_word.get(predicted_index)

        #  avoid repeating same word continuously
        if word and word != last_word:
            result.append(word)
            last_word = word

    print("Predicted Title Words:", " ".join(result),'.')

generate_title("AI guide for beginners studying abroad")